# CSE 4310 
# Programming Assignment 3 (P3)

### Task 1 (30 points) 

Implement Hough transform from scratch to detect circles in the image. Refer to this wiki page for pseudocode(https://en.wikipedia.org/wiki/Circle_Hough_Transform). Superimpose detected circles on the original image.

Note: You shouldn't use cv2.HoughCircles() to detect the circles.

<img src = Images/coins.png>

In [8]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

importedCoins = cv2.imread("Images/coins.png", cv2.IMREAD_GRAYSCALE)

In [13]:
# We first need to do edge detection. This will be so much better now since I can just directly call cv2.Canny
blurred_coins = cv2.GaussianBlur(importedCoins, (5, 5), 1)

edges = cv2.Canny(blurred_coins, 50, 150)

# plt.imshow(edges, cmap="gray") 
# Checked to see that the Canny Edge Detection worked :D

In [14]:
# Now even though this is slow, I will make r a parameter, so thus, our accumulator array will be 3 dimensional. 
# The reason that I'm doing this is because I am too lazy to do fine-tuning and finding all the radius for the respective coins.

# Firstly I need to make the accumulator array.
height, width = edges.shape

r_min = 10
r_max = 100

A = np.zeros(
    (height, width, r_max - r_min + 1),
    dtype=np.uint32
)

angles = np.deg2rad(np.arange(0, 360))
# Obtain all edge coordinates
ys, xs = np.nonzero(edges)

# Originally this used no gradients, but since there is a lot of calculations, so we will use those here.
# ... which now means I have to calculate the direction vector
Gx = cv2.Sobel(blurred_coins, cv2.CV_32F, 1, 0, ksize=3)
Gy = cv2.Sobel(blurred_coins, cv2.CV_32F, 0, 1, ksize=3)

ys, xs = np.nonzero(edges)

# Gradient vectors at the edge pixels
gx = Gx[ys, xs]
gy = Gy[ys, xs]

magnitude = np.sqrt(gx**2 + gy**2)

# Avoid division by zero (cool way AI showed me to do this)
valid = magnitude > 1e-6

xs = xs[valid]
ys = ys[valid]
gx = gx[valid]
gy = gy[valid]
magnitude = magnitude[valid]

# Normalize the gradient vectors
nx = gx / magnitude
ny = gy / magnitude

# Now we populate the accumulator array. "radius_small and radius_big"
for index, r in enumerate(range(r_min, r_max + 1)):
    for direction in [-1, 1]:

        a = np.rint(xs + direction * r * nx).astype(int)
        b = np.rint(ys + direction * r * ny).astype(int)

        valid_centers = (
            (a >= 0) & (a < width) &
            (b >= 0) & (b < height)
        )

        np.add.at(
            A[:, :, index],
            (b[valid_centers], a[valid_centers]),
            1
        )


### Task 2 (30 points)

Implement Harris corner detector from scratch. Mark detected corners on original image. 

Note: You shouldn't use cv2.cornerHarris() to detect the corners.

<img src = Images/checkerboard.jpg>

### Task 3 (40 points)

Implement multiscale template matching that measures the SSD differences (normalized for brightness but NOT for contrast) between the template and each location of the image at multiple scales. Both image and template are grayscale images. For this task, consider a linear range of scales, np.arange( 0.5, 3, .25). Show your best match for each size(0.5, 0.75, 1 etc.) in a 2X5 plot. 

Template: <img src = Images/template.png>

Test Image: <img src = Images/test_image.png style="width:537px; height:400px">

Hint: function multiscale_correlation (in the slides) has many similarities with the function that you are asked to implement. The only difference is that multiscale_correlation computes scores based on normalized cross-correlation, whereas your scores are based on SSD differences after brightness normalization.

Note: 
1. You should implement template matching from scratch, and not use the opencv template matching.
2. Only use "bilinear interpolation" when resizing.

## Submission Guidelines:

    Submit through Canvas your source code in a single .ipynb file. The name of the .ipynb file should be YourStudentID.ipynb. (For example: 1001234567.ipynb)
    Import all the images from the ./Images directory. Your TA will use the same directory name to grade your submission.
    You don't need to attach the image file with your submission.
